In [1]:
from res_opt_core import EnergyModel, Battery, RenewableProduction, Grid, AuctionMarket, Load, plot_battery_operation
import numpy as np
import pandas as pd

In [4]:
# Test data - 24 hours
time_start = pd.Timestamp("2023-01-01 00:00:00", tz="Europe/Zurich")
time_end = pd.Timestamp("2023-01-02 00:00:00", tz="Europe/Zurich")
time_idx = pd.date_range(start=time_start, end=time_end, freq="1h", inclusive="left")

df_input = pd.DataFrame(
    index=time_idx,
    data={
        # Price: high morning/evening, low lunch (PV), low night
        "da_price": [
            0.13, 0.1, 0.1, 0.15, 0.15, 0.15,      # 00-05: Night (low)
            0.17, 0.18, 0.2, 0.2, 0.18, 0.15,     # 06-11: Morning peak, then drops
            0.14, 0.12, 0.13, 0.15, 0.2, 0.22,      # 12-17: Lunch dip (PV), then rises
            0.28, 0.25, 0.2, 0.2, 0.19, 0.17      # 18-23: Evening peak, then drops
        ],
        # PV: only during daylight (08-17)
        "pv_production": [
            0, 0, 0, 0, 0, 0,            # 00-05: No sun
            0, 0, 2, 5, 8, 10,           # 06-11: Sun rising
            12, 10, 8, 5, 2, 0,          # 12-17: Sun setting
            0, 0, 0, 0, 0, 0             # 18-23: No sun
        ],
        # Load: high morning/evening, moderate midday
        "load": [
            3, 2, 2, 2, 3, 4,            # 00-05: Low night load
            6, 8, 9, 8, 7, 6,            # 06-11: Morning peak
            5, 5, 4, 5, 6, 7,            # 12-17: Moderate
            9, 10, 9, 7, 5, 4            # 18-23: Evening peak
        ],
    },
)


df_input

,da_price,pv_production,load
2023-01-01 00:00:00+01:00,0.13,0,3
2023-01-01 01:00:00+01:00,0.10,0,2
2023-01-01 02:00:00+01:00,0.10,0,2
2023-01-01 03:00:00+01:00,0.15,0,2
2023-01-01 04:00:00+01:00,0.15,0,3
2023-01-01 05:00:00+01:00,0.15,0,4
2023-01-01 06:00:00+01:00,0.17,0,6
2023-01-01 07:00:00+01:00,0.18,0,8
2023-01-01 08:00:00+01:00,0.20,2,9
2023-01-01 09:00:00+01:00,0.20,5,8


In [ ]:
# Model setup

model = EnergyModel(
    num_steps=len(df_input.index),
    slot_length="hour",
    solver="scip",
)

# Components
battery = Battery(
    name="bess",
    energy_capacity=20.0, #kWh
    power_nominal=10.0, #kW
    eta=0.922, # efficiency one way
    soc_final=None
)

pv = RenewableProduction(
    name="pv",
    power_max=df_input["pv_production"], #kW
    allow_spill=True, # excess production can be spilled/curtailed
    cost_spill=0.2 # cost of curtailment
)

load = Load(
    name="load",
    load_curve=df_input['load'],
    lost_load_allowed=True,
    value_of_lost_load=1000.0
)

dynamischer_tariff = AuctionMarket(
    name='dynamic_tariff',
    price_curve=df_input['da_price'],
    #market_time_unit='1hr' definier resolutoin of market 
)

grid = Grid(
    name="grid",
    assets=[battery, pv, load],
    markets=[dynamischer_tariff],
    power_min=-2.0, #kW -> bezug aus netz
)

model.add_component(battery)
model.add_component(pv)
model.add_component(grid)
model.add_component(dynamischer_tariff)
model.add_component(load)
# model.add_component(boiler)

#list_of_components = [pv, battery, grid]
#model.add_component(list_of_components)

model.build_and_run(silent=False) #scip outputs visible

df_results = model.results.timeseries_to_pandas()
df_results

SCIP version 10.0.0 [precision: 8 byte] [memory: block] [mode: optimized] [LP solver: SoPlex 8.0.0] [GitHash: 0c80fdd8e9]
Copyright (c) 2002-2025 Zuse Institute Berlin (ZIB)

External libraries: 
  SoPlex 8.0.0         Linear programming solver developed at Zuse Institute Berlin (soplex.zib.de) [GitHash: 2207cfb2]
  CppAD 20180000.0     Algorithmic Differentiation of C++ algorithms developed by B. Bell (github.com/coin-or/CppAD)
  ZLIB 1.3.1           General purpose compression library by J. Gailly and M. Adler (zlib.net)
  MPFR 4.2.1           GNU Multiple Precision Floating-Point Reliable Library (mpfr.org)
  Boost 1.88.0         Boost C++ Libraries (boost.org)
  TinyCThread 1.2      small portable implementation of the C11 threads API (tinycthread.github.io)
  GMP 6.3.0            GNU Multiple Precision Arithmetic Library developed by T. Granlund (gmplib.org)
  ZIMPL 3.7.0          Zuse Institute Mathematical Programming Language developed by T. Koch (zimpl.zib.de)
  AMPL/MP 4.0.3 

,bess__var_power,bess__var_power_reserve_up,bess__var_power_reserve_down,bess__var_soc_slot_start,bess__var_soc_slot_end,bess__var_min_energy_violation,bess__var_max_energy_violation,bess__var_power_charge,bess__var_power_discharge,bess__var_is_charging,...,pv__var_power,pv__var_power_reserve_up,pv__var_power_reserve_down,pv__var_spill,dynamic_tariff__var_power,load__var_power,load__var_power_reserve_up,load__var_power_reserve_down,load__var_lost_load,timestamp
timestamp,,,,,,,,,,,,,,,,,,,,,
2000-01-01 00:00:00,0.000000,0.0,0.0,0.500000,5.000000e-01,0.0,0.0,0.000000,0.00,1.0,...,0.0,0.0,0.0,0.0,-2.000000,-2.00,0.0,0.0,1.00,2000-01-01 00:00:00
2000-01-01 01:00:00,0.000000,0.0,0.0,0.500000,5.000000e-01,0.0,0.0,0.000000,0.00,1.0,...,0.0,0.0,0.0,0.0,-2.000000,-2.00,0.0,0.0,0.00,2000-01-01 01:00:00
2000-01-01 02:00:00,0.000000,0.0,0.0,0.500000,5.000000e-01,0.0,0.0,0.000000,0.00,1.0,...,0.0,0.0,0.0,0.0,-2.000000,-2.00,0.0,0.0,0.00,2000-01-01 02:00:00
2000-01-01 03:00:00,0.000000,0.0,0.0,0.500000,5.000000e-01,0.0,0.0,0.000000,0.00,1.0,...,0.0,0.0,0.0,0.0,-2.000000,-2.00,0.0,0.0,0.00,2000-01-01 03:00:00
2000-01-01 04:00:00,0.000000,0.0,0.0,0.500000,5.000000e-01,0.0,0.0,0.000000,0.00,1.0,...,0.0,0.0,0.0,0.0,-2.000000,-2.00,0.0,0.0,1.00,2000-01-01 04:00:00
2000-01-01 05:00:00,0.000000,0.0,0.0,0.500000,5.000000e-01,0.0,0.0,0.000000,0.00,1.0,...,0.0,0.0,0.0,0.0,-2.000000,-2.00,0.0,0.0,2.00,2000-01-01 05:00:00
2000-01-01 06:00:00,0.000000,0.0,0.0,0.500000,5.000000e-01,0.0,0.0,0.000000,0.00,1.0,...,0.0,0.0,0.0,0.0,-2.000000,-2.00,0.0,0.0,4.00,2000-01-01 06:00:00
2000-01-01 07:00:00,3.220000,0.0,0.0,0.500000,3.253796e-01,0.0,0.0,0.000000,3.22,0.0,...,0.0,0.0,0.0,0.0,-2.000000,-5.22,0.0,0.0,2.78,2000-01-01 07:00:00
2000-01-01 08:00:00,5.000000,0.0,0.0,0.325380,5.422993e-02,0.0,0.0,0.000000,5.00,0.0,...,2.0,0.0,0.0,0.0,-2.000000,-9.00,0.0,0.0,0.00,2000-01-01 08:00:00


In [1]:
plot_battery_operation(df_results.index, battery, {"dynamischer_tariff": df_input['da_price']})

NameError: name 'plot_battery_operation' is not defined